# Registration — learn a deformation between two images

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fideus-labs/KonfAI/blob/main/examples/Registration/Registration_demo.ipynb)

**Run all the cells.** This notebook trains a diffeomorphic `VoxelMorph` to align a `MOVING` image
onto a `FIXED` one, on **real pelvis CT slices**, then measures how much of the deformation it
recovered.

The deformation is one we apply, so the answer is known exactly and the registered output can be
checked numerically. For registration between two *different patients* — a real anatomical difference,
scored on reference segmentations — see [`examples/ImpactReg`](../ImpactReg/).

Expect roughly **2 to 3 minutes on a GPU**.

In [ ]:
# Setup: find KonfAI (cloning it on Colab), install what is missing, load the notebook helpers.
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    REPO_DIR = Path("/content/KonfAI")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/fideus-labs/KonfAI", str(REPO_DIR)], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples").is_dir())
sys.path.insert(0, str(REPO_DIR / "examples"))

from konfai_demo import latest_checkpoint, read, run, setup, show

EXAMPLE_DIR, DATASET_DIR, DEVICE = setup(REPO_DIR, "Registration", ("konfai", f"{REPO_DIR}[imaging]"), "huggingface_hub", "matplotlib", "scipy")

## 1. The data

`make_dataset.py` takes six axial slices from each of the five public pelvis CT cases, windows them
to `[0, 1]`, and crops each to `256x256` around the body. `FIXED` is the slice as acquired; `MOVING`
is the same slice pushed through a smooth displacement field of up to 8 voxels that the script picks
— so the task has a ground-truth answer while the anatomy is real.

The CT comes from the same public subset the segmentation example uses, cached by the Hub after the
first run.

In [ ]:
run(sys.executable, "make_dataset.py")

CASES = sorted(path.name for path in DATASET_DIR.iterdir() if path.is_dir())
print(len(CASES), "cases:", ", ".join(CASES))

## 2. Look at one pair

The difference map is where the two disagree — mostly the edges of bone and muscle, displaced by the
field. That is what the network has to undo.

In [ ]:
import numpy as np

case = DATASET_DIR / CASES[0]
fixed, moving = read(case / "FIXED.mha")[0], read(case / "MOVING.mha")[0]
print(case.name, "| shape:", fixed.shape, "| mean |FIXED - MOVING|:", f"{np.abs(fixed - moving).mean():.4f}")

SCALE = (0.0, float(np.abs(fixed - moving).max()))  # reused below, so before and after compare by eye
show([
    ("FIXED — a real CT slice", fixed, "gray", (0.0, 1.0)),
    ("MOVING — the same slice, deformed", moving, "gray", (0.0, 1.0)),
    ("|FIXED - MOVING|", np.abs(fixed - moving), "magma", SCALE),
])

## 3. Train, predict, evaluate

| Command | Config | What it does |
|---|---|---|
| `konfai TRAIN` | `Config.yml` | trains `VoxelMorph` on an MSE image-similarity loss |
| `konfai PREDICTION` | `Prediction.yml` | writes the registered image as `MOVED.mha` per case |
| `konfai EVALUATION` | `Evaluation.yml` | scores `MOVED` **and** `MOVING` against `FIXED`, so the JSON contains the before *and* the after |

`VoxelMorph` takes two inputs and the order of the groups is load-bearing: the first `is_input` group
(`FIXED`) is branch `0`, the second (`MOVING`) is branch `1`.

In [ ]:
run("konfai", "TRAIN", "-y", *DEVICE, "--config", "Config.yml")

run("konfai", "PREDICTION", "-y", *DEVICE, "--config", "Prediction.yml", "--models", latest_checkpoint("REG_BASELINE"))
run("konfai", "EVALUATION", "-y", "--config", "Evaluation.yml")

## 4. The result

`MOVED:FIXED:*` is the error **after** registration, `MOVING:FIXED:*` the error **before**. A
successful run leaves the first well below the second.

In [ ]:
import json

import numpy as np

metrics = json.loads((EXAMPLE_DIR / "Evaluations" / "REG_BASELINE" / "Metric_TRAIN.json").read_text())
for metric in ("MAE", "MSE"):
    before = metrics["aggregates"][f"MOVING:FIXED:{metric}"]["mean"]
    after = metrics["aggregates"][f"MOVED:FIXED:{metric}"]["mean"]
    print(f"{metric}: {before:.4f} before  ->  {after:.4f} after   ({before / after:.1f}x better)")

moved = read(EXAMPLE_DIR / "Predictions" / "REG_BASELINE" / "Dataset" / case.name / "MOVED.mha")[0]
show([
    ("|FIXED - MOVING|  before", np.abs(fixed - moving), "magma", SCALE),
    ("|FIXED - MOVED|  after (same scale)", np.abs(fixed - moved), "magma", SCALE),
    ("MOVED", moved, "gray", (0.0, 1.0)),
])

## What to change next

- **a harder deformation** — raise `AMPLITUDE` in `make_dataset.py`, or point `dataset_filenames` at
  your own `FIXED` / `MOVING` pairs.
- **more training** — `epochs: 400` is about 30 seconds of GPU time per 100 epochs here, and the
  scheduler `step_size` is stretched to match; both live in `Config.yml`.
- **a better loss** — this example optimises image similarity alone; real deformable registration adds
  a smoothness regulariser on the deformation field and often swaps MSE for normalised cross-correlation.
- **no training at all, and no known answer** — `examples/ImpactReg` aligns two *different* patients
  with published presets (FireANTs, ConvexAdam, elastix) and scores the result by propagating one
  patient's reference labels through the recovered field.

`README.md` in this folder explains the two-input wiring and the metric table.